# Création de personnage — POC
## 1. Chargement et nettoyage
Rejouer les cellules dans l'ordre. Données : `../data/personnages_fr.json`.

### 1.1 Imports et session Spark

In [1]:
import numpy as np
import pandas as pd
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Creation_perso")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

### 1.2 Chargement, nettoyage, drapeau `free`
Interrupteur : décommenter la ligne `df_perso = ...filter(~free)` pour exclure les free des graphiques. Le moteur de création, lui, ignorera toujours les free.

In [2]:
CHEMIN_JSON = "../data/personnages_fr.json"          # adapte si besoin

IDS_FREE = [
    130114003, 130114008, 130114015, 130114018, 130124003, 130124009, 130124010, 130124026, 130124027,
    130124030, 130124034, 130124045, 130124054, 130134005, 130144008, 130144016, 130144022, 130214011,
    130324016, 130324021, 130344003, 130344007, 130344008, 130344010, 130344012, 130454001, 130454002,
    130454005, 130454007, 130454009, 130454011, 130454012, 130454013, 130454015, 130464005, 130524003,
    130724002,
]

SLOTS_COMPETENCES = ["competence_1", "competence_2", "competence_3"]
SLOTS_TOUS = SLOTS_COMPETENCES + ["incarnation"]

df_raw = spark.read.option("multiLine", True).json(CHEMIN_JSON)
df_perso = df_raw.select(F.explode("personnages").alias("perso")).select("perso.*")      # on retire l'enveloppe
df_perso = df_perso.drop("nom_en", "nom_jp", "stats_lvl_100", "description")             # EN/JP, doublon, texte long

def nettoyer_slot(slot):
    s = F.col(f"competences.{slot}")
    return s.withField("niveaux", F.transform(s["niveaux"], lambda n: n.dropFields("nom_jp", "description_jp")))

df_perso = (df_perso
            .withColumn("competences", F.struct(*[nettoyer_slot(s).alias(s) for s in SLOTS_TOUS]))
            .withColumn("free", F.col("id").isin(IDS_FREE)))

df_perso_complet = df_perso                                    # référence : tous les persos
df_perso = df_perso_complet.filter(~F.col("free"))           # interrupteur : SANS les free

print(f"{df_perso_complet.count()} persos | dont free : {df_perso_complet.filter('free').count()}")

218 persos | dont free : 37


### 1.3 Table des effets `df_eff` (compétences 1-3 + incarnation)

In [3]:
SPLIT = r"(?<=\.)\s+(?=[A-ZÀÂÉÈÊÎÔÛÇ])"

# Phrases déjà traitées ailleurs (Break, Charge, Mode enhance, Enhance, Heal) ou sans intérêt ici
EXCLUS = (r"^(Brise la Volonté|Charge la Volonté de|Entre soi-même en mode|Capacité d'un allié|Soins HP"
          r"|Attaque (tranchante|magique|perçante|de frappe)( de Volonté)?( à puissance énorme)? sur (un ennemi|tous les ennemis)$"
          r"|Temps de préparation|Rotation|\（仮\）|$)")

REGLES = [
    (r"^Résistances? physique( et magique)? -N%", "Malus résistance physique"),
    (r"^Résistance magique (sur l'ennemi )?-N%|^Résistances? physique et magique -N%"
     r"|^Résistance physique -N% et résistance magique -N%", "Malus résistance magique"),
    (r"^Résistance à l'élément [^ ]+ -N%|^La résistance à l'élément [^ ]+ diminue"
     r"|et de N% la résistance à l'élément", "Malus résistance d'élément"),
    (r"^De plus, DEF et MDEF -N% si l'ennemi", "Malus DEF et MDEF (ennemi d'un élément)"),
    (r"^ATK -N%", "Malus ATK"),
    (r"^MATK -N%", "Malus MATK"),
    (r"^MP -N% pour l'élément", "Malus MP selon l'élément"),
    (r"^Élément [^ ]+ -N% sur l'ennemi", "Malus d'élément sur l'ennemi"),
    (r"^Réduit de N% le Break de Volonté de l'ennemi", "Réduction du Break ennemi"),
    (r"^Réduit de N% le Break de la compétence ennemie", "Réduction du Break de la compétence ennemie"),
    (r"^Réduit (de N% )?la Charge de Volonté (ennemie|de l'ennemi)|^Charge de Volonté de l'ennemi -N%", "Réduction de la Charge ennemie"),
    (r"^Inflige des dégâts continus", "Dégâts continus"),
    (r"^Provoque l'ennemi", "Provocation"),
    (r"^Inflige une contre-attaque", "Contre-attaque"),
    (r"^(Guérit|Soigne|Guérison)", "Guérison / soin de statut"),
    (r"^Restauration de MP", "Restauration de MP (alliés)"),
    (r"^Absorption des HP", "Absorption de HP"),
    (r"^Multiplie (sa|le|la) |^Capacité à charger", "Multiplication Charge/Break (soi)"),
    (r"^Attaque .*N% de dégâts de HP maximum", "Attaque avec coût en HP"),
    (r"^Et attaque ", "Attaque supplémentaire"),
    (r"^En contrepartie|^Résistance [^ ]+ sur soi-même -N%", "Malus en contrepartie"),
]

BONUS = [
    (r"(?<![A-Za-z])ATK", "Bonus ATK"),
    (r"MATK", "Bonus MATK"),
    (r"[Rr]ésistances? physique", "Bonus résistance physique"),
    (r"[Rr]ésistances? (physique et )?magique", "Bonus résistance magique"),
    (r"[Rr]ésistance (d'|à l')élément", "Bonus résistance d'élément"),
    (r"[Dd]égâts critiques", "Bonus dégâts critiques"),
    (r"Taux de coup critique", "Bonus taux de critique"),
    (r"Taux d'esquive", "Bonus esquive"),
]

def normaliser(col):
    c = F.regexp_replace(col, "’", "'")
    c = F.regexp_replace(c, r"tour\(s\)", "tours")
    c = F.regexp_replace(c, r"\s*\([^()]*(tours|maximum)[^()]*\)", "")
    c = F.regexp_replace(c, r"\s*\((faible|moyenne|forte|énorme)\)", "")
    c = F.regexp_replace(c, r"\d+(,\d+)?", "N")
    c = F.regexp_replace(c, r"\s+", " ")
    c = F.regexp_replace(c, r"[\s.,;]+$", "")
    return F.trim(c)

def construire_effets(df_source, slots):
    """Une ligne par (perso, compétence, effet), à partir de la description du niveau max de chaque slot."""
    def phrases_du_slot(slot):
        c = F.col(f"competences.{slot}")
        return df_source.select(F.col("id").alias("perso_id"), "element", F.lit(slot).alias("slot"),
                                F.element_at(c["niveaux"], -1)["description"].alias("description"))

    df_phr = (reduce(lambda a, b: a.unionByName(b), [phrases_du_slot(s) for s in slots])
              .select("perso_id", "element", "slot", F.explode(F.split(F.trim("description"), SPLIT)).alias("phrase"))
              .withColumn("p", normaliser(F.col("phrase")))
              .filter(~F.col("p").rlike(EXCLUS)))

    col_p = F.col("p")                      # jamais une variable nommée « p » : on garde un nom explicite
    vide = F.array().cast("array<string>")

    def correspondances(regles):
        return F.filter(F.array(*[F.when(col_p.rlike(r), F.lit(n)) for r, n in regles]), lambda x: x.isNotNull())

    est_rf = col_p.rlike(r"^Recollection Field")
    rf_valeur = F.regexp_extract(F.col("phrase"), r"^Recollection Field(?: spécial)? \([^)]*\)\s*\+(\d+)", 1)
    etiq_rf = (F.when(col_p.rlike(r"^Recollection Field spécial"),
                      F.array(F.concat(F.lit("Recollection Field spécial +"), rf_valeur)))
                .when(est_rf, F.array(F.concat(F.lit("Recollection Field +"), rf_valeur)))
                .otherwise(vide))

    est_boost = ~est_rf & col_p.rlike(
        r"[Aa]ugmentation (de N% )?(des dégâts(?! critiques)|de la Charge de Volonté|de la capacité à briser|du Break de Volonté)"
        r"|^Augmente les dégâts(?! critiques)|^Lance sur tous les alliés un bonus augmentant les dégâts")
    effet_boost = (F.when(col_p.rlike(r"[Aa]ugmentation (de N% )?des dégâts(?! critiques)|^Augmente les dégâts(?! critiques)|bonus augmentant les dégâts"), "Dégâts")
                    .when(col_p.rlike(r"[Aa]ugmentation de la Charge"), "Charge de Volonté")
                    .otherwise("Break de Volonté"))
    condition = (F.when(col_p.contains("subissant des dégâts continus"), "ennemi en dégâts continus")
                  .when(col_p.contains("subissant un malus de résistance"),
                        F.concat(F.lit("ennemi avec malus de résistance "),
                                 F.regexp_extract(col_p, r"malus de résistance\s*(.*)$", 1)))
                  .when(col_p.contains("Recollection Field"), "Recollection Field atteint un seuil")
                  .when(col_p.contains("lien de compétences"), "lien de compétences")
                  .when(col_p.contains("nombre de cibles"), "nombre de cibles")
                  .when(col_p.rlike(r"Volonté de l'ennemi (dépasse|est de N% ou plus)"), "Volonté ennemie au-dessus d'un seuil")
                  .when(col_p.rlike(r"(quand|lorsque) le bonus"), "bonus d'ATK ou de critique atteint un seuil")
                  .otherwise("autre condition"))
    etiq_boost = F.when(est_boost, F.array(F.concat(effet_boost, F.lit(" + : "), condition))).otherwise(vide)

    cible = (F.when(col_p.contains("soi-même"), " (soi-même)")
              .when(col_p.rlike("alliés?"), " (alliés)")
              .otherwise(" (cible non précisée)"))
    est_bonus = ~est_boost & ~est_rf & col_p.rlike(
        r"\+N%|en augmentant de N%|en augmentant l'(ATK|MATK) de N%"
        r"|^Augmentation de N% des dégâts critiques|^Augmente les dégâts critiques de N%")
    etiq_bonus = F.when(est_bonus, F.transform(correspondances(BONUS), lambda n: F.concat(n, cible))).otherwise(vide)

    toutes = F.concat(etiq_rf, etiq_boost, correspondances(REGLES), etiq_bonus)
    df_eff = (df_phr
              .withColumn("effets", F.when(F.size(toutes) == 0, F.array(F.lit("Autre"))).otherwise(toutes))
              .withColumn("effet", F.explode("effets"))
              .drop("effets"))

    autres = df_eff.filter(F.col("effet") == "Autre").select("p").distinct().collect()
    print(f"{slots} : {df_eff.select('effet').distinct().count()} effets distincts | "
          f"phrases non classées (« Autre ») : {len(autres)}")
    for r in autres:
        print("   non classée :", r["p"])
    return df_eff

df_eff = construire_effets(df_perso_complet, SLOTS_TOUS)      # défaut retenu : avec l'incarnation

['competence_1', 'competence_2', 'competence_3', 'incarnation'] : 59 effets distincts | phrases non classées (« Autre ») : 0


### 1.4 Table de travail (stats niveau 100 + kit 0/1) et contrôles
Effectifs : ne pas interpréter les effets à moins de 10 persos.

In [4]:
STATS = ["HP", "MP", "ATK", "MATK", "DEF", "MDEF", "CRIT", "SPD"]
GRILLE_100 = {"HP": 75, "DEF": 15, "MDEF": 15, "ATK": 25, "MATK": 25}

def table_de_travail(base, effets_long):
    """base : 1 ligne par perso (index perso_id). effets_long : colonnes perso_id, effet."""
    base = base.copy()
    base["famille"] = np.where(base["ATK"] >= base["MATK"], "physique", "magique")
    cats = base[["c1", "c2", "c3"]].fillna("")
    roles = pd.DataFrame({
        "Rôle Break": cats.apply(lambda ligne: any("BREAK" in x for x in ligne), axis=1),
        "Rôle Charge": cats.apply(lambda ligne: any("CHARGE" in x for x in ligne), axis=1),
        "Rôle Mode enhance": cats.apply(lambda ligne: any(x == "ENHANCE_MODE" for x in ligne), axis=1)})
    effets = ((pd.crosstab(effets_long["perso_id"], effets_long["effet"]) > 0)      # True si le perso a l'effet
          .reindex(base.index, fill_value=False))
    kit = pd.concat([roles, effets], axis=1).astype(bool)
    base["taille_kit"] = effets.sum(axis=1)
    return dict(base=base, kit=kit)

def preparer_table(df_source, df_effets, niveau=100):
    cols = [F.col(f"stats.lvl_{niveau}.{s}").alias(s) for s in STATS]
    base = (df_source.select(F.col("id").alias("perso_id"), "nom_complet", "element", "arme", "free",
                             F.col("competences.competence_1.categorie").alias("c1"),
                             F.col("competences.competence_2.categorie").alias("c2"),
                             F.col("competences.competence_3.categorie").alias("c3"), *cols)
            .toPandas().set_index("perso_id"))
    effets_long = df_effets.select("perso_id", "effet").distinct().toPandas()
    return table_de_travail(base, effets_long)

def controle_table(tab):
    base, kit = tab["base"], tab["kit"]
    print("Persos :", len(base), "| colonnes du kit :", kit.shape[1])
    print("\nEffectifs free x famille :")
    print(pd.crosstab(base["free"], base["famille"]))
    ref = base[~base["free"]]
    print(f"\nSur la grille (niveau 100), non-free, n = {len(ref)} :")
    for s, pas in GRILLE_100.items():
        print(f"  {s:5} {(ref[s] % pas == 0).sum():3} / {len(ref)}")
    print("\nEffectif par effet/rôle du kit (non-free), du plus rare au plus fréquent :")
    print(kit.loc[ref.index].sum().sort_values().to_string())

tab = preparer_table(df_perso_complet, df_eff, niveau=100)
controle_table(tab)

Persos : 218 | colonnes du kit : 62

Effectifs free x famille :
famille  magique  physique
free                      
False         64       117
True          16        21

Sur la grille (niveau 100), non-free, n = 181 :
  HP    132 / 181
  DEF   135 / 181
  MDEF  134 / 181
  ATK   123 / 181
  MATK  143 / 181

Effectif par effet/rôle du kit (non-free), du plus rare au plus fréquent :
Restauration de MP (alliés)                                                   0
Malus en contrepartie                                                         0
Recollection Field +3                                                         0
Absorption de HP                                                              1
Break de Volonté + : ennemi avec malus de résistance physique ou magique      1
Bonus ATK (cible non précisée)                                                1
Bonus taux de critique (soi-même)                                             1
Charge de Volonté + : ennemi avec malus de résistance

### 2. Moteur


In [5]:
from scipy.stats import multivariate_normal

STATS_MENU = ["MP", "CRIT", "SPD"]            # pas de grille : on choisit parmi les valeurs déjà vues

def ajuster_moteur_stats(tab):
    """Apprend, sur les persos non free, une loi normale à 8 dimensions PAR FAMILLE (physique / magique)."""
    ref = tab["base"][~tab["base"]["free"]]
    modeles = {}
    for fam, groupe in ref.groupby("famille"):
        modeles[fam] = dict(moy=groupe[STATS].mean().values,
                            cov=np.cov(groupe[STATS].values, rowvar=False),
                            poids=len(groupe) / len(ref), n=len(groupe))
    return dict(ref=ref, modeles=modeles,
                menus={s: np.sort(ref[s].unique()) for s in STATS_MENU},
                bornes_obs=ref[STATS].agg(["min", "max"]),
                moy=ref[STATS].mean(), std=ref[STATS].std(ddof=0))

def arrondir_stats(moteur, X):
    X = X.copy()
    for j, s in enumerate(STATS):
        if s in GRILLE_100:
            X[:, j] = np.round(X[:, j] / GRILLE_100[s]) * GRILLE_100[s]
        else:
            menu = moteur["menus"][s]
            X[:, j] = menu[np.abs(X[:, [j]] - menu[None, :]).argmin(axis=1)]
    return X

def lire_contraintes(moteur, contraintes):
    """contraintes = {"ATK": {"egal": 2800}, "SPD": {"pct_min": 80}, "HP": {"min": 7500, "max": 8500}, "famille": "physique"}"""
    ref = moteur["ref"]
    bornes, famille = {}, None
    for cle, regle in (contraintes or {}).items():
        if cle == "famille":
            if regle not in ("physique", "magique"):
                raise ValueError("famille doit valoir 'physique' ou 'magique'")
            famille = regle
            continue
        if cle not in STATS:
            raise ValueError(f"stat inconnue : {cle} (attendu : {STATS} ou 'famille')")
        b = dict(egal=None, min=-np.inf, max=np.inf)
        for mot, v in regle.items():
            if mot == "egal":      b["egal"] = float(v)
            elif mot == "min":     b["min"] = max(b["min"], float(v))
            elif mot == "max":     b["max"] = min(b["max"], float(v))
            elif mot == "pct_min": b["min"] = max(b["min"], np.percentile(ref[cle], v))
            elif mot == "pct_max": b["max"] = min(b["max"], np.percentile(ref[cle], v))
            else: raise ValueError(f"{cle} : règle inconnue « {mot} » (egal, min, max, pct_min, pct_max)")
        if b["egal"] is not None and cle in GRILLE_100:
            pas = GRILLE_100[cle]
            arrondi = round(b["egal"] / pas) * pas
            if arrondi != b["egal"]:
                print(f"  {cle} = {b['egal']:.0f} n'est pas sur la grille (pas {pas}) : arrondi à {arrondi:.0f}")
            b["egal"] = float(arrondi)
        if b["min"] > b["max"] or (b["egal"] is not None and not b["min"] <= b["egal"] <= b["max"]):
            raise ValueError(f"{cle} : contraintes contradictoires {regle}")
        bornes[cle] = b
    if len([s for s in bornes if bornes[s]["egal"] is not None]) >= len(STATS):
        raise ValueError("Laisse au moins une stat libre.")
    return bornes, famille

def _reels_conformes(moteur, bornes, famille):
    """Nombre de VRAIS persos non free qui respectent déjà les contraintes (« egal » = à un pas près)."""
    ref = moteur["ref"]
    ok = pd.Series(True, index=ref.index)
    for s, b in bornes.items():
        tol = GRILLE_100.get(s, moteur["std"][s] * 0.1)
        if b["egal"] is not None:
            ok &= (ref[s] - b["egal"]).abs() <= tol
        ok &= (ref[s] >= b["min"]) & (ref[s] <= b["max"])
    if famille:
        ok &= ref["famille"] == famille
    return int(ok.sum())

def _tirer(moteur, bornes, famille, n_tirages, rng, temperature=1.0):
    idx = {s: i for i, s in enumerate(STATS)}
    fixes = [idx[s] for s, b in bornes.items() if b["egal"] is not None]
    libres = [i for i in range(len(STATS)) if i not in fixes]
    xf = np.array([bornes[STATS[i]]["egal"] for i in fixes])
    familles = [famille] if famille else list(moteur["modeles"])
    cond, logw = {}, []
    for fam in familles:
        m = moteur["modeles"][fam]
        moy, cov = m["moy"], m["cov"]
        if fixes:                                       # conditionnement gaussien
            Sff, Srf, Srr = cov[np.ix_(fixes, fixes)], cov[np.ix_(libres, fixes)], cov[np.ix_(libres, libres)]
            K = Srf @ np.linalg.inv(Sff)
            cond[fam] = (moy[libres] + K @ (xf - moy[fixes]), Srr - K @ Srf.T)
            logw.append(np.log(m["poids"]) + multivariate_normal(moy[fixes], Sff).logpdf(xf))
        else:
            cond[fam] = (moy, cov)
            logw.append(np.log(m["poids"]))
    w = np.exp(np.array(logw) - max(logw)); w /= w.sum()   # quelle famille rend ces valeurs fixées les plus plausibles
    choix = rng.choice(len(familles), size=n_tirages, p=w)
    X = np.zeros((n_tirages, len(STATS)))
    fam_tiree = np.empty(n_tirages, dtype=object)
    for k, fam in enumerate(familles):
        masque = choix == k
        mu, sigma = cond[fam]
        X[np.ix_(masque, libres)] = rng.multivariate_normal(mu, sigma * temperature ** 2, size=int(masque.sum()))
        fam_tiree[masque] = fam
    if fixes:
        X[:, fixes] = xf
    return X, fam_tiree

def proposer_stats(moteur, contraintes=None, n=5, graine=0, n_tirages=20000, temperature=1.0, distance_min=None):
    rng = np.random.default_rng(graine)
    bornes, famille = lire_contraintes(moteur, contraintes)
    n_reels = _reels_conformes(moteur, bornes, famille)
    X, fam_tiree = _tirer(moteur, bornes, famille, n_tirages, rng, temperature)
    X = arrondir_stats(moteur, X)
    marge = max(temperature - 1, 0) * moteur["std"]          # température > 1 : on tolère un peu au-delà des plages observées

    ok = np.ones(n_tirages, dtype=bool)
    for j, s in enumerate(STATS):
        b = bornes.get(s)
        if b is not None and b["egal"] is not None:
            continue                                                    # valeur imposée : pas de test de plage
        ok &= (X[:, j] >= moteur["bornes_obs"].loc["min", s] - marge[s]) & (X[:, j] <= moteur["bornes_obs"].loc["max", s] + marge[s])
        if b is not None:
            ok &= (X[:, j] >= b["min"]) & (X[:, j] <= b["max"])
    fam_obs = np.where(X[:, STATS.index("ATK")] >= X[:, STATS.index("MATK")], "physique", "magique")
    ok &= fam_obs == fam_tiree                                          # la famille visée doit rester vraie après arrondi
    if famille:
        ok &= fam_obs == famille

    if distance_min is not None:                                        # nouveauté minimale : on écarte les copies de l'existant
        Zref = ((moteur["ref"][STATS] - moteur["moy"]) / moteur["std"]).values / np.sqrt(len(STATS))
        idx_ok = np.flatnonzero(ok)
        Zc = ((X[idx_ok] - moteur["moy"].values) / moteur["std"].values) / np.sqrt(len(STATS))
        dmin = np.sqrt(((Zc[:, None, :] - Zref[None, :, :]) ** 2).sum(axis=2)).min(axis=1)
        ok[idx_ok[dmin < distance_min]] = False
    print(f"Tirages acceptés : {ok.sum()} / {n_tirages} ({ok.mean():.1%}) | "
          f"vrais persos non free qui respectent déjà les contraintes : {n_reels}")
    if n_reels < 10:
        print("  ⚠ moins de 10 vrais persos : la proposition s'appuie surtout sur le modèle, pas sur des exemples.")
    uniques = np.unique(X[ok], axis=0)
    if len(uniques) == 0:
        raise ValueError("Aucun tirage ne respecte ces contraintes : relâche-en une (ou élargis un min/max).")
    choisis = uniques[rng.permutation(len(uniques))[:n]]

    res = pd.DataFrame(choisis, columns=STATS).astype(int)
    res["famille"] = np.where(res["ATK"] >= res["MATK"], "physique", "magique")
    # plus proche vrai perso (même échelle que construire_espace : stats centrées-réduites)
    Zref = ((moteur["ref"][STATS] - moteur["moy"]) / moteur["std"]).values / np.sqrt(len(STATS))
    Zres = ((res[STATS] - moteur["moy"]) / moteur["std"]).values / np.sqrt(len(STATS))
    D = np.sqrt(((Zres[:, None, :] - Zref[None, :, :]) ** 2).sum(axis=2))
    res["plus_proche"] = moteur["ref"]["nom_complet"].values[D.argmin(axis=1)]
    res["distance"] = D.min(axis=1).round(3)
    return res

def percentiles_stats(moteur, res):
    """Rang percentile (0-100) de chaque stat par rapport aux persos non free."""
    ref = moteur["ref"]
    return pd.DataFrame({s: [(ref[s] <= v).mean() * 100 for v in res[s]] for s in STATS}, index=res.index).round(0).astype(int)

def valider_moteur_stats(moteur, n=5000, graine=1):
    """Sans contrainte : les stats générées ressemblent-elles aux stats réelles ?"""
    rng = np.random.default_rng(graine)
    ref = moteur["ref"]
    X, fam_t = _tirer(moteur, {}, None, n, rng)
    X = arrondir_stats(moteur, X)
    dans = np.all((X >= moteur["bornes_obs"].loc["min"].values) & (X <= moteur["bornes_obs"].loc["max"].values), axis=1)
    gen = pd.DataFrame(X[dans], columns=STATS)
    print(f"Tirages gardés (dans la plage observée) : {len(gen)} / {n}")
    print("\nMoyenne / écart-type : réel vs généré")
    print(pd.DataFrame({"moy réel": ref[STATS].mean(), "moy généré": gen.mean(),
                        "écart-type réel": ref[STATS].std(), "écart-type généré": gen.std()}).round(0).astype(int).to_string())
    diff = (gen.corr() - ref[STATS].corr()).abs().values
    print(f"\nÉcart de corrélation réel vs généré : moyen {diff[np.triu_indices(8,1)].mean():.2f} | max {diff.max():.2f}")
    part_phys = (gen["ATK"] >= gen["MATK"]).mean()
    print(f"Part de physiques : réel {(ref['famille']=='physique').mean():.1%} | généré {part_phys:.1%}")
    Zr = ((ref[STATS] - moteur["moy"]) / moteur["std"]).values / np.sqrt(8)
    Zg = ((gen[STATS].iloc[:1000] - moteur["moy"]) / moteur["std"]).values / np.sqrt(8)
    Dgr = np.sqrt(((Zg[:, None, :] - Zr[None, :, :]) ** 2).sum(axis=2)).min(axis=1)
    Drr = np.sqrt(((Zr[:, None, :] - Zr[None, :, :]) ** 2).sum(axis=2)); np.fill_diagonal(Drr, np.inf)
    print(f"Distance au plus proche vrai perso : généré {np.median(Dgr):.3f} | entre vrais persos {np.median(Drr.min(axis=1)):.3f}")
    copies = (np.array([(ref[STATS].values == row).all(axis=1).any() for row in gen[STATS].values[:1000]])).mean()
    print(f"Part des persos générés identiques (8 stats) à un vrai perso : {copies:.1%}")

In [6]:
moteur = ajuster_moteur_stats(tab)
print({fam: (m["n"], round(m["poids"], 3)) for fam, m in moteur["modeles"].items()})
valider_moteur_stats(moteur)

{'magique': (64, 0.354), 'physique': (117, 0.646)}
Tirages gardés (dans la plage observée) : 4633 / 5000

Moyenne / écart-type : réel vs généré
      moy réel  moy généré  écart-type réel  écart-type généré
HP        7804        7822              571                524
MP         273         273               14                 14
ATK       2702        2700              294                285
MATK      2552        2557              303                294
DEF       1504        1506               94                 90
MDEF      1492        1491              100                 94
CRIT       734         733               44                 43
SPD        331         330                7                  7

Écart de corrélation réel vs généré : moyen 0.02 | max 0.05
Part de physiques : réel 64.6% | généré 63.7%
Distance au plus proche vrai perso : généré 0.495 | entre vrais persos 0.411
Part des persos générés identiques (8 stats) à un vrai perso : 0.0%


## Création de stats : mode d'emploi

Tu décris ce que tu veux dans le dictionnaire `CONTRAINTES`. Le moteur propose des persos de **niveau 100** dont les stats respectent tes contraintes et ressemblent à ce que font les développeurs.
Tu peux ne mettre qu'une seule contrainte, ou aucune (`CONTRAINTES = {}` : le moteur tire un perso « typique »).

### Les règles disponibles (pour chacune des 8 stats : HP, MP, ATK, MATK, DEF, MDEF, CRIT, SPD)

| Règle | Sens | Exemple |
|---|---|---|
| `egal` | valeur exacte | `"ATK": {"egal": 2800}` |
| `min` | au moins cette valeur | `"HP": {"min": 7500}` |
| `max` | au plus cette valeur | `"HP": {"max": 8500}` |
| `pct_min` | au moins ce **percentile** (0 à 100) des persos non free | `"SPD": {"pct_min": 80}` : parmi les 20 % les plus rapides |
| `pct_max` | au plus ce percentile | `"CRIT": {"pct_max": 30}` : parmi les 30 % les plus bas |
| `"famille"` | `"physique"` (ATK ≥ MATK) ou `"magique"` | `"famille": "magique"` |

On peut combiner plusieurs règles sur la même stat : `"HP": {"min": 7500, "max": 8500}`.
Un **percentile** est un rang : 80 signifie « meilleur que 80 % des vrais persos ». Il est calculé sur les 181 persos non free.

### Exemples à copier dans `CONTRAINTES`

```python
# 1. Une valeur exacte (le reste est déduit)
{"ATK": {"egal": 2800}}

# 2. Une fourchette
{"HP": {"min": 7500, "max": 8500}}

# 3. Un perso parmi les plus rapides
{"SPD": {"pct_min": 80}}

# 4. Un magique rapide
{"famille": "magique", "SPD": {"pct_min": 80}}

# 5. Un « tank » : bon en HP, DEF et MDEF
{"HP": {"pct_min": 75}, "DEF": {"pct_min": 75}, "MDEF": {"pct_min": 75}}

# 6. Un canon de verre physique
{"famille": "physique", "ATK": {"min": 3000}, "CRIT": {"pct_min": 80}}
```

### Ce que le moteur fait de tes contraintes

- **Grille** : pour HP, ATK, MATK, DEF et MDEF, une valeur `egal` hors grille est arrondie (ATK 2810 devient 2800) et un message le signale.
  Les pas sont : HP 75, DEF 15, MDEF 15, ATK 25, MATK 25.
- **MP, CRIT, SPD** : les valeurs proposées sont des valeurs déjà vues chez les vrais persos.
- **Stats non contraintes** : elles sont déduites des corrélations réelles (par exemple, plus la DEF est haute, plus le HP a tendance à l'être aussi).
- **Au moins une stat libre** : on ne peut pas fixer les 8 stats avec `egal`.

### Comment lire le résultat

1. **« Tirages acceptés »** : la part des tirages du moteur qui respectent tes contraintes. Un taux faible (quelques %) veut dire que tes contraintes visent une zone rare.
2. **« Vrais persos qui respectent déjà les contraintes »** : si c'est **moins de 10**, la proposition vient du modèle et pas d'exemples réels. Elle est plausible, pas prouvée.
3. **`plus_proche` et `distance`** : le vrai perso le plus semblable. Chez les vrais persos, la distance au plus proche voisin vaut en médiane **0,41**, et 95 % sont sous **0,78**. Au-delà de 0,8, le perso est atypique.
4. **Table des percentiles** : le rang de chaque stat (90 = dans les 10 % les plus hauts). Elle permet de voir d'un coup d'œil le « profil » du perso.

### Quand ça ne marche pas

- `ValueError : Aucun tirage ne respecte ces contraintes` : tes contraintes sont trop sévères ensemble (essai : `HP min 9400` et `DEF min 1900`). Relâche-en une.
- `contraintes contradictoires` : un `min` supérieur à un `max` sur la même stat.
- Pour obtenir d'autres propositions avec les mêmes contraintes, change `graine` (`graine=1`, `2`, ...) ou `n` (nombre de propositions).

In [7]:
CONTRAINTES = {
    "ATK": {"egal": 3000 },          # valeur exacte (arrondie à la grille si besoin)
    #"SPD": {"pct_min": 80},         # au moins le 80e percentile des persos non free
    #"famille": "physique",          # ou "magique" ; omets la ligne pour laisser le moteur choisir
     "HP": {"min": 7200, "max": 7900},
    # "CRIT": {"pct_max": 30},
}

proposition = proposer_stats(moteur, CONTRAINTES, n=5, graine=0, distance_min=0.6)

# stats : température (à utiliser avec prudence, T <= 1.3)
#proposition = proposer_stats(moteur, CONTRAINTES, n=5, graine=0, temperature=1.2)


display(proposition)
display(percentiles_stats(moteur, proposition))

Tirages acceptés : 1823 / 20000 (9.1%) | vrais persos non free qui respectent déjà les contraintes : 4
  ⚠ moins de 10 vrais persos : la proposition s'appuie surtout sur le modèle, pas sur des exemples.


,HP,MP,ATK,MATK,DEF,MDEF,CRIT,SPD,famille,plus_proche,distance
0,7500,281,3000,2900,1305,1350,730,335,physique,[Vent du champ glacé] Eugeo,0.743
1,7500,300,3000,2425,1515,1485,817,329,physique,[La mémoire de la Rose Bleue] Eugeo,0.662
2,7875,291,3000,2500,1410,1500,815,330,physique,"[Ce jour-là, au pied de l'osmanthe] Eydis",0.645
3,7725,294,3000,2525,1350,1605,791,330,physique,[La rêveuse qui a le vent en poupe] Yuriko Nanao,0.911
4,7875,274,3000,2225,1695,1485,832,330,physique,[Hidden Resolve] Mito,0.667


,HP,MP,ATK,MATK,DEF,MDEF,CRIT,SPD
0,30,71,86,82,1,10,51,74
1,30,99,86,46,55,42,97,44
2,55,96,86,55,19,48,96,45
3,45,98,86,56,3,91,90,45
4,55,46,86,9,98,42,99,45


### Moteur kit

In [8]:
import re, difflib

SEUIL_RARE = 10      # un effet présent chez moins de 10 persos non free est regroupé avec des effets proches

def categorie_effet(e):
    if e.startswith("Rôle"): return "Rôle"
    if e.startswith("Recollection Field"): return "Recollection Field"
    if " + : " in e: return "Augmentation conditionnelle"
    if e.startswith("Bonus"): return "Bonus (buff)"
    if e.startswith(("Malus", "Réduction")) and e != "Malus en contrepartie": return "Malus sur l'ennemi"
    return "Autres mécaniques"

def famille_effet(e):
    """Niveau 2 du regroupement : effets rares proches. None = pas de famille intermédiaire."""
    cat = categorie_effet(e)
    if cat == "Augmentation conditionnelle":
        return "Dégâts + (conditions rares)" if e.startswith("Dégâts +") else "Break/Charge + (conditions rares)"
    if cat == "Bonus (buff)":
        return "Bonus résistances (rares)" if "résistance" in e else "Bonus offensifs (rares)"
    return None

def construire_groupes(K, seuil=SEUIL_RARE):
    """Associe chaque effet à un groupe : lui-même s'il est fréquent, sinon sa famille, sinon sa catégorie."""
    n = K.sum()
    groupe, rares = {}, []
    for e in K.columns:
        if n[e] >= seuil or categorie_effet(e) in ("Rôle", "Recollection Field"):
            groupe[e] = e
        else:
            rares.append(e)
    fam = {e: famille_effet(e) for e in rares}
    nfam = {f: int(K[[e for e in rares if fam[e] == f]].any(axis=1).sum()) for f in set(fam.values()) if f}
    for e in rares:
        f = fam[e]
        groupe[e] = f if (f and nfam[f] >= seuil) else "Rares : " + categorie_effet(e)
    return groupe

def _generer_matrice(G, tailles, k, a, n, rng, depart=(), temperature=1.0):
    cols = G.shape[1]
    sorties = np.zeros((n, cols), dtype=bool)
    for i in range(n):
        taille = max(int(rng.choice(tailles)), len(depart)); deja = list(depart)
        while len(deja) < taille:
            w, _n = _poids_suivant(G, deja, k, a, temperature)
            if w.sum() <= 0: break
            deja.append(int(rng.choice(cols, p=w / w.sum())))
        sorties[i, deja] = True
    return sorties

def _calibrer(G, tailles, k, n=3000, iterations=8, graine=0):
    """Ajuste un poids par groupe pour que les fréquences générées égalent les fréquences réelles."""
    rng = np.random.default_rng(graine)
    cible = G.values.mean(axis=0)
    a = np.ones(G.shape[1])
    for _ in range(iterations):
        freq = _generer_matrice(G, tailles, k, a, n, rng).mean(axis=0)
        a = a * (np.clip(cible, 1e-3, None) / np.clip(freq, 1e-3, None)) ** 0.7
    return a

def _calibrer_conditionnel(moteur, r, pool, n=1500, iterations=8, graine=0):
    """Même calibrage, mais pour une demande précise : cible = fréquences réelles chez les persos qui ont déjà les éléments imposés (lissées)."""
    if not r:
        return moteur["a"]
    G, k = moteur["G"], moteur["k"]
    Gv = G.values
    masque = Gv[:, r].all(axis=1)
    n_m = int(masque.sum())
    f_all = Gv.mean(axis=0)
    f_m = Gv[masque].mean(axis=0) if n_m > 0 else f_all
    cible = (n_m * f_m + k * f_all) / (n_m + k)
    a = moteur["a"].copy()
    rng = np.random.default_rng(graine)
    for _ in range(iterations):
        freq = _generer_matrice(G, pool, k, a, n, rng, depart=r).mean(axis=0)
        maj = (np.clip(cible, 1e-3, None) / np.clip(freq, 1e-3, None)) ** 0.7
        maj[r] = 1.0
        a = a * maj
    return a

def ajuster_moteur_kit(tab, seuil=SEUIL_RARE, k_lissage=5):
    base, kit = tab["base"], tab["kit"]
    K = kit.loc[~base["free"]]
    K = K.loc[:, K.sum() > 0]                                   # les effets que seuls les free ont disparaissent
    groupe = construire_groupes(K, seuil)
    noms = sorted(set(groupe.values()))
    G = pd.DataFrame({g: K[[e for e in groupe if groupe[e] == g]].any(axis=1) for g in noms})
    membres = {g: K[[e for e in groupe if groupe[e] == g]].sum() for g in noms}      # effectif de chaque effet dans son groupe
    tailles = G.sum(axis=1).values
    a = _calibrer(G, tailles, k_lissage)
    return dict(K=K, G=G, groupe=groupe, membres=membres, k=k_lissage, tailles=tailles, a=a)

def _poids_suivant(G, deja, k, a=None, temperature=1.0):
    """Probabilité de chaque groupe d'être le prochain, sachant ce qu'on a déjà choisi (lissée vers la fréquence globale)."""
    Gv = G.values
    freq_all = Gv.mean(axis=0)
    masque = Gv[:, deja].all(axis=1) if deja else np.ones(len(Gv), dtype=bool)
    n_s = masque.sum()
    freq_s = Gv[masque].mean(axis=0) if n_s > 0 else np.zeros(Gv.shape[1])
    w = (n_s * freq_s + k * freq_all) / (n_s + k)
    if a is not None:
        w = w * a                                              # calibrage : corrige le biais du tirage sans remise
    if temperature != 1.0:
        w = np.where(w > 0, w, 0) ** (1.0 / temperature)       # température > 1 : on aplatit (les choix rares gagnent des chances)
    w[deja] = 0
    return w, int(n_s)

def _verifier_noms(moteur, effets):
    inconnus = [e for e in effets if e not in moteur["groupe"]]
    if inconnus:
        aide = {e: difflib.get_close_matches(e, list(moteur["groupe"]), n=3, cutoff=0.4) for e in inconnus}
        raise ValueError(f"Effet inconnu (ou jamais présent chez un non-free) : {aide}")

def proposer_kit(moteur, imposes=(), n=5, graine=0, temperature=1.0):
    """imposes : liste de noms d'effets ou de rôles (noms exacts de moteur['groupe'])."""
    rng = np.random.default_rng(graine)
    imposes = list(imposes)
    _verifier_noms(moteur, imposes)
    G, cols = moteur["G"], list(moteur["G"].columns)
    r = [cols.index(g) for g in sorted({moteur["groupe"][e] for e in imposes})]       # groupes imposés
    masque = G.values[:, r].all(axis=1) if r else np.ones(len(G), dtype=bool)
    n_match = int(masque.sum())
    pool = moteur["tailles"][masque] if n_match >= 10 else moteur["tailles"][moteur["tailles"] >= len(r)]
    if len(pool) == 0:
        pool = np.array([len(r)])
    print(f"Vrais persos non free qui ont déjà tous les éléments imposés : {n_match}"
          + ("   ⚠ moins de 10 : lissage fort vers les fréquences globales" if 0 < n_match < 10 else "")
          + ("   ⚠ aucun : combinaison jamais vue" if n_match == 0 and r else ""))
    a_q = _calibrer_conditionnel(moteur, r, pool)
    kits = []
    for _ in range(n):
        taille = max(int(rng.choice(pool)), len(r))
        deja = list(r)
        while len(deja) < taille:
            w, _n = _poids_suivant(G, deja, moteur["k"], a_q, temperature)
            if w.sum() <= 0: break
            deja.append(int(rng.choice(len(cols), p=w / w.sum())))
        kit = set(imposes)
        for j in deja:
            g = cols[j]
            if any(moteur["groupe"][e] == g for e in imposes):
                continue                                      # groupe déjà couvert par un effet imposé
            m = moteur["membres"][g]; m = m[m > 0]
            kit.add(str(rng.choice(m.index, p=(m / m.sum()).values)))
        kits.append(sorted(kit))
    return kits
def afficher_kits(kits):
    for i, kit in enumerate(kits):
        print(f"Kit {i}  ({len(kit)} éléments)")
        for e in kit: print("   -", e)

def valider_moteur_kit(moteur, n=3000, graine=1):
    """Sans contrainte : les kits générés ressemblent-ils aux kits réels ? (comparaison au niveau des groupes)"""
    rng = np.random.default_rng(graine)
    G, cols = moteur["G"], list(moteur["G"].columns)
    Gen = pd.DataFrame(_generer_matrice(G, moteur["tailles"], moteur["k"], moteur["a"], n, rng), columns=cols)
    f_r, f_g = G.mean(), Gen.mean()
    print(f"Fréquence de chaque groupe, réel vs généré : écart moyen {np.abs(f_r - f_g).mean()*100:.1f} points | max {np.abs(f_r - f_g).max()*100:.1f} points")
    cr, cg = G.astype(float).corr().values, Gen.astype(float).corr().values
    iu = np.triu_indices(len(cols), 1)
    print(f"Liens entre groupes (corrélations) : écart moyen {np.nanmean(np.abs(cr - cg)[iu]):.3f} | corrélation entre les deux matrices {np.corrcoef(np.nan_to_num(cr)[iu], np.nan_to_num(cg)[iu])[0,1]:.2f}")
    print(f"Taille du kit : réel moyenne {G.sum(axis=1).mean():.2f} (écart-type {G.sum(axis=1).std():.2f}) | généré {Gen.sum(axis=1).mean():.2f} ({Gen.sum(axis=1).std():.2f})")
    reels = {tuple(r) for r in G.values.astype(int)}
    copies = np.mean([tuple(r) in reels for r in Gen.values.astype(int)])
    print(f"Part des kits générés identiques à un vrai kit (au niveau des groupes) : {copies:.1%}")
    return Gen

def choisir_lissage(moteur, valeurs=(0.5, 1, 2, 5, 10, 20, 1e9), n_ordres=3, graine=0):
    """Validation croisée « un perso à la fois » : le lissage k prédit-il bien les effets d'un perso qu'il n'a pas vu ?"""
    rng = np.random.default_rng(graine)
    G = moteur["G"]; Gv = G.values; m = Gv.shape[1]
    res = {}
    for k in valeurs:
        total, compte = 0.0, 0
        for i in range(len(Gv)):
            Gi = pd.DataFrame(np.delete(Gv, i, axis=0))
            vrais = list(np.flatnonzero(Gv[i]))
            for _ in range(n_ordres):
                ordre = list(rng.permutation(vrais)); deja = []
                for j in ordre:
                    w, _n = _poids_suivant(Gi, deja, k)
                    total += np.log(w[j] / w.sum()); compte += 1; deja.append(int(j))
        res[k] = total / compte
    out = pd.Series(res, name="log-vraisemblance moyenne (plus haut = mieux)")
    out.index = [("infini (fréquences globales)" if k >= 1e8 else k) for k in out.index]
    return out

def verifier_conditionnel(moteur, imposes, n=2000, graine=2, afficher=8):
    """Quand on impose des effets, les AUTRES éléments bougent-ils comme chez les vrais persos qui les ont ?"""
    rng = np.random.default_rng(graine)
    imposes = list(imposes); _verifier_noms(moteur, imposes)
    G, cols = moteur["G"], list(moteur["G"].columns)
    r = [cols.index(g) for g in sorted({moteur["groupe"][e] for e in imposes})]
    masque = G.values[:, r].all(axis=1)
    pool = moteur["tailles"][masque] if masque.sum() >= 10 else moteur["tailles"][moteur["tailles"] >= len(r)]
    a_q = _calibrer_conditionnel(moteur, r, pool)
    gen = pd.DataFrame(_generer_matrice(G, pool, moteur["k"], a_q, n, rng, depart=r), columns=cols)
    tab_ = pd.DataFrame({"global %": G.mean() * 100, "réel (avec) %": G[masque].mean() * 100, "généré %": gen.mean() * 100})
    tab_ = tab_.drop(index=[cols[j] for j in r])
    tab_["écart réel"] = tab_["réel (avec) %"] - tab_["global %"]
    tab_["écart généré"] = tab_["généré %"] - tab_["global %"]
    print(f"Vrais persos avec {imposes} : {int(masque.sum())}")
    if masque.sum() >= 10:
        print(f"Les écarts à la normale vont dans le même sens (corrélation réel/généré) : {tab_['écart réel'].corr(tab_['écart généré']):.2f}")
    top = tab_.reindex(tab_["écart réel"].abs().sort_values(ascending=False).index).head(afficher)
    print(top[["global %", "réel (avec) %", "généré %"]].round(0).astype(int).to_string())

def lister_effets(moteur):
    """Tous les noms utilisables dans `imposes`, avec leur effectif chez les non-free et leur groupe."""
    n = moteur["K"].sum()
    t = pd.DataFrame({"effet": n.index, "persos": n.values, "groupe": [moteur["groupe"][e] for e in n.index]})
    t["catégorie"] = t["effet"].map(categorie_effet)
    return t.sort_values(["catégorie", "persos"], ascending=[True, False]).reset_index(drop=True)

In [9]:
moteur_kit = ajuster_moteur_kit(tab)
print(f"{moteur_kit['G'].shape[1]} groupes pour {moteur_kit['K'].shape[1]} effets/rôles\n")
display(choisir_lissage(moteur_kit))      # k = 5 doit être le meilleur
print()
_ = valider_moteur_kit(moteur_kit)

30 groupes pour 59 effets/rôles



0.5                            -3.106429
1.0                            -3.054654
2.0                            -3.027348
5.0                            -3.022846
10.0                           -3.027671
20.0                           -3.034613
infini (fréquences globales)   -3.072944
Name: log-vraisemblance moyenne (plus haut = mieux), dtype: float64


Fréquence de chaque groupe, réel vs généré : écart moyen 0.5 points | max 2.1 points
Liens entre groupes (corrélations) : écart moyen 0.066 | corrélation entre les deux matrices 0.70
Taille du kit : réel moyenne 4.83 (écart-type 1.68) | généré 4.82 (1.64)
Part des kits générés identiques à un vrai kit (au niveau des groupes) : 3.7%


## Création du kit : mode d'emploi

Tu listes dans `IMPOSES` les effets ou rôles que tu veux **absolument** avoir. Le moteur complète le reste d'un kit cohérent avec ce que font les développeurs.
`IMPOSES = []` donne un kit « typique ».

### Les noms valides
Utilise des noms **exacts**. La cellule `display(lister_effets(moteur_kit))` les affiche tous, avec leur effectif.
Une faute de frappe donne une erreur avec des suggestions (par exemple `"Degats continus"` suggère `"Dégâts continus"`).

### Exemples (à copier dans `IMPOSES`)

```python
# 1. Un seul effet
["Dégâts continus"]

# 2. Un rôle et un effet
["Rôle Break", "Malus résistance magique"]

# 3. Un effet rare (moins de 10 persos) : possible, mais peu d'exemples réels
["Provocation"]

# 4. Une combinaison jamais vue : le moteur la fabrique mais te prévient
["Recollection Field +2", "Recollection Field spécial +3"]
```

### Comment lire le résultat
1. **« Vrais persos qui ont déjà tous les éléments imposés : N »** : le nombre d'exemples réels. Moins de 10 : le moteur lisse fortement vers les fréquences globales. 0 : combinaison jamais vue.
2. **Les kits** : chaque kit est une liste d'effets. Leur taille suit celle des vrais kits (médiane 5).
3. **`verifier_conditionnel(moteur_kit, IMPOSES)`** : pour tes contraintes, compare les fréquences réelles et générées. Les deux colonnes doivent être proches quand N est d'au moins 10.

### Limites
- Les effets rares sont regroupés : le moteur tire un groupe, puis un effet précis dedans selon sa fréquence réelle.
- Il n'y a pas de notion de **slot** (compétence 1, 2, 3 ou incarnation) : le kit est une liste d'effets pour tout le perso.
- Un kit proposé est plausible, pas optimisé.

### Température : T
- T = 1 reproduit le moteur normal
- T < 1 donne des persos plus typiques
- T > 1 des persos plus inhabituels.

In [10]:
display(lister_effets(moteur_kit))        # les noms valides (à commenter une fois que tu les connais)

IMPOSES = ["Dégâts continus", "Malus résistance magique"]

kits = proposer_kit(moteur_kit, IMPOSES, n=5, graine=0, temperature=1.5)
afficher_kits(kits)
verifier_conditionnel(moteur_kit, IMPOSES)

,effet,persos,groupe,catégorie
0,Dégâts + : ennemi avec malus de résistance phy...,40,Dégâts + : ennemi avec malus de résistance phy...,Augmentation conditionnelle
1,Dégâts + : ennemi en dégâts continus,36,Dégâts + : ennemi en dégâts continus,Augmentation conditionnelle
2,Dégâts + : Recollection Field atteint un seuil,24,Dégâts + : Recollection Field atteint un seuil,Augmentation conditionnelle
3,Dégâts + : bonus d'ATK ou de critique atteint ...,15,Dégâts + : bonus d'ATK ou de critique atteint ...,Augmentation conditionnelle
4,Dégâts + : lien de compétences,8,Dégâts + (conditions rares),Augmentation conditionnelle
5,Charge de Volonté + : Recollection Field attei...,6,Break/Charge + (conditions rares),Augmentation conditionnelle
6,Dégâts + : Volonté ennemie au-dessus d'un seuil,5,Dégâts + (conditions rares),Augmentation conditionnelle
7,Charge de Volonté + : ennemi en dégâts continus,4,Break/Charge + (conditions rares),Augmentation conditionnelle
8,Dégâts + : ennemi avec malus de résistance d'é...,4,Dégâts + (conditions rares),Augmentation conditionnelle
9,Dégâts + : nombre de cibles,4,Dégâts + (conditions rares),Augmentation conditionnelle


Vrais persos non free qui ont déjà tous les éléments imposés : 3   ⚠ moins de 10 : lissage fort vers les fréquences globales
Kit 0  (5 éléments)
   - Attaque supplémentaire
   - Bonus ATK (alliés)
   - Charge de Volonté + : ennemi en dégâts continus
   - Dégâts continus
   - Malus résistance magique
Kit 1  (3 éléments)
   - Dégâts continus
   - Malus résistance magique
   - Recollection Field +2
Kit 2  (5 éléments)
   - Attaque supplémentaire
   - Dégâts continus
   - Malus résistance magique
   - Réduction du Break ennemi
   - Rôle Break
Kit 3  (5 éléments)
   - Bonus résistance magique (alliés)
   - Dégâts + : ennemi en dégâts continus
   - Dégâts continus
   - Malus résistance magique
   - Rôle Break
Kit 4  (3 éléments)
   - Dégâts continus
   - Malus MP selon l'élément
   - Malus résistance magique
Vrais persos avec ['Dégâts continus', 'Malus résistance magique'] : 3
                                   global %  réel (avec) %  généré %
Attaque supplémentaire                    9    

### Validation du perso


In [11]:
import json, os
from itertools import combinations

def creer_persos(moteur, moteur_kit, contraintes_stats=None, imposes_kit=(), n=5, graine=0,
                 temperature_stats=1.0, distance_min=None, temperature_kit=1.0):
    """Assemble stats (moteur stats) et kit (moteur kit) : les deux sont indépendants dans les vraies données."""
    print("--- stats ---")
    df = proposer_stats(moteur, contraintes_stats, n=n, graine=graine, temperature=temperature_stats, distance_min=distance_min)
    print("--- kit ---")
    kits = proposer_kit(moteur_kit, imposes_kit, n=n, graine=graine, temperature=temperature_kit)
    return [dict(stats={s: int(df.loc[i, s]) for s in STATS}, kit=list(kits[i])) for i in range(min(len(df), len(kits)))]

def afficher_persos(persos):
    for i, p in enumerate(persos):
        fam = "physique" if p["stats"]["ATK"] >= p["stats"]["MATK"] else "magique"
        print(f"Perso {i} ({fam}) : " + " | ".join(f"{s} {p['stats'][s]}" for s in STATS))
        print("   kit : " + " ; ".join(p["kit"]))

def valider_perso(moteur, moteur_kit, perso, seuil_distance=0.78):
    """Diagnostic d'un perso (généré ou imaginé par toi) : stats, kit, puis synthèse."""
    ref, stats, kit = moteur["ref"], perso["stats"], list(perso["kit"])
    alertes = []
    # ---------- STATS ----------
    lignes = []
    for s in STATS:
        v = stats[s]
        lo, hi = moteur["bornes_obs"].loc["min", s], moteur["bornes_obs"].loc["max", s]
        if s in GRILLE_100: grille = "✓" if v % GRILLE_100[s] == 0 else f"✗ (pas {GRILLE_100[s]})"
        else:               grille = "✓ vue" if v in set(ref[s]) else "jamais vue"
        lignes.append(dict(stat=s, valeur=v, percentile=int(round((ref[s] <= v).mean() * 100)),
                           plage_observée="✓" if lo <= v <= hi else f"✗ ({int(lo)}-{int(hi)})", grille=grille))
        if not lo <= v <= hi: alertes.append(f"{s} = {v} hors de la plage vue chez les vrais persos ({int(lo)} à {int(hi)})")
        if grille.startswith("✗"): alertes.append(f"{s} = {v} hors grille (pas {GRILLE_100[s]}) : vrai pour 21 à 32 % des vrais persos selon la stat")
    print("STATS")
    print(pd.DataFrame(lignes).to_string(index=False))
    z = lambda d: ((pd.Series(d)[STATS] - moteur["moy"]) / moteur["std"]).values / np.sqrt(len(STATS))
    Zref = ((ref[STATS] - moteur["moy"]) / moteur["std"]).values / np.sqrt(len(STATS))
    D = np.sqrt(((Zref - z(stats)) ** 2).sum(axis=1)); j = D.argmin()
    print(f"Famille : {'physique' if stats['ATK'] >= stats['MATK'] else 'magique'} | plus proche vrai perso : {ref['nom_complet'].iloc[j]} (distance {D[j]:.2f} ; "
          f"vrais persos : médiane 0,41, 95e percentile {seuil_distance})")
    if D[j] == 0: alertes.append("stats identiques à un vrai perso")
    elif D[j] > seuil_distance: alertes.append(f"stats atypiques : distance {D[j]:.2f} au plus proche vrai perso (plus que 95 % des vrais)")

    # ---------- KIT ----------
    _verifier_noms(moteur_kit, kit)
    G, K, groupe = moteur_kit["G"], moteur_kit["K"], moteur_kit["groupe"]
    cols = list(G.columns); Gv = G.values; N = len(Gv)
    gk = sorted({groupe[e] for e in kit})
    x = np.zeros(len(cols), dtype=bool); x[[cols.index(g) for g in gk]] = True
    dk = 1 - (Gv & x).sum(axis=1) / np.maximum((Gv | x).sum(axis=1), 1); jk = dk.argmin()
    inter = Gv.astype(int) @ Gv.astype(int).T; un = Gv.sum(1)[:, None] + Gv.sum(1)[None, :] - inter
    dd = 1 - inter / np.maximum(un, 1); np.fill_diagonal(dd, 9)
    nn_ref = dd.min(axis=1)
    print("\nKIT")
    lignes = [dict(élément=e, vrais_persos_avec=int(K[e].sum()), groupe_de_calcul=groupe[e] if groupe[e] != e else "(lui-même)") for e in kit]
    print(pd.DataFrame(lignes).to_string(index=False))
    taille = len(gk); tailles = G.sum(axis=1)
    print(f"Taille (groupes) : {taille} | vrais kits : médiane {int(tailles.median())}, de {int(tailles.min())} à {int(tailles.max())} "
          f"(percentile {int(round((tailles <= taille).mean() * 100))})")
    nom_proche = ref.loc[G.index[jk], "nom_complet"]
    print(f"Kit réel le plus proche : {nom_proche} (distance de Jaccard {dk[jk]:.2f} ; entre vrais kits : médiane {np.median(nn_ref):.2f}, 95e percentile {np.quantile(nn_ref, .95):.2f})")
    rares = [e for e in kit if K[e].sum() < 10]
    if rares: alertes.append(f"effets présents chez moins de 10 vrais persos : {rares}")
    if dk[jk] == 0: alertes.append(f"kit identique (au niveau des groupes) à celui de {nom_proche}")
    elif dk[jk] > np.quantile(nn_ref, .95): alertes.append(f"kit atypique : distance {dk[jk]:.2f}, plus que 95 % des vrais kits")
    if taille > tailles.max() or taille < tailles.min(): alertes.append(f"taille de kit ({taille}) hors de ce qu'on voit chez les vrais persos")
    for a, b in combinations(gk, 2):
        na, nb, co = int(G[a].sum()), int(G[b].sum()), int((G[a] & G[b]).sum())
        if co == 0 and na * nb / N >= 3:
            alertes.append(f"« {a} » et « {b} » ne sont jamais ensemble chez les vrais persos (attendu ≈ {na * nb / N:.1f} si indépendants)")
    # ---------- SYNTHÈSE ----------
    print("\nSYNTHÈSE")
    if not alertes: print("  ✓ rien d'anormal par rapport aux persos existants")
    for a in alertes: print("  ⚠", a)
    print("  (stats et kit sont indépendants dans tes données : on ne vérifie pas de lien entre les deux)")
    return alertes

def calculer_ratios_niveaux(df_source):
    """Rapports niveau 1 / 80 / 120 sur niveau 100, estimés sur les persos non free."""
    cols = [F.col(f"stats.lvl_{lv}.{s}").alias(f"{s}_{lv}") for lv in (1, 80, 100, 120) for s in STATS]
    t = df_source.filter(~F.col("free")).select(*cols).toPandas()
    exacts = {1: 1 / 5, 80: 0.8, 120: 1.2}                  # règle exacte pour HP, ATK, MATK, DEF, MDEF
    return {lv: {s: (exacts[lv] if s in GRILLE_100 else float(np.median(t[f"{s}_{lv}"] / t[f"{s}_100"]))) for s in STATS}
            for lv in (1, 80, 120)}

def stats_tous_niveaux(stats100, ratios):
    out = {f"lvl_{lv}": {s: int(round(stats100[s] * ratios[lv][s])) for s in STATS} for lv in (1, 80)}
    out["lvl_100"] = {s: int(stats100[s]) for s in STATS}
    out["lvl_120"] = {s: int(round(stats100[s] * ratios[120][s])) for s in STATS}
    return {k: out[k] for k in ("lvl_1", "lvl_80", "lvl_100", "lvl_120")}

def exporter_perso(perso, moteur, df_source, id_perso, nom, titre, element, arme, chemin="../data/personage_new.json",
                   description="", parametres=None, remplacer=False):
    """Crée le fichier s'il n'existe pas, sinon ajoute le perso. Les textes de compétences restent à écrire à la main."""
    elements, armes = sorted(moteur["ref"]["element"].unique()), sorted(moteur["ref"]["arme"].unique())
    if element not in elements: raise ValueError(f"element inconnu : {element} (attendu : {elements})")
    if arme not in armes: raise ValueError(f"arme inconnue : {arme} (attendue : {armes})")
    ids_orig = {r["id"] for r in df_source.select("id").collect()}
    if id_perso in ids_orig: raise ValueError(f"id {id_perso} déjà utilisé dans le jeu : choisis-en un autre")
    ratios = calculer_ratios_niveaux(df_source)
    stats = stats_tous_niveaux(perso["stats"], ratios)
    emplacement = lambda slot: dict(nom="", type="", categorie="", incarnation=(slot == "incarnation"),
                                    niveaux=[dict(niveau=1, id=None, nom="", description="", mp=None, puissance=None)])
    nouveau = dict(
        id=int(id_perso), nom=nom, nom_complet=f"{titre} {nom}", traduction_fr_officielle=False, titre=titre,
        nom_en="", nom_jp="", element=element, arme=arme, rarete=4,
        niveaux_paliers={"eveil_0": 80, "eveil_max": 100, "niveau_max": 120},
        stats=stats, stats_lvl_100=stats["lvl_100"], regen_mp_taux=500, description=description,
        competences={s: emplacement(s) for s in ("competence_1", "competence_2", "competence_3", "incarnation")},
        label=f"character.nouveau_{id_perso}",
        genere=True, kit_propose=list(perso["kit"]), parametres_generation=parametres or {})
    if os.path.exists(chemin):
        with open(chemin, encoding="utf-8") as f: contenu = json.load(f)
    else:
        contenu = dict(source="Persos générés par l'outil de création (POC)", nb_personnages=0, personnages=[])
    deja = [i for i, p in enumerate(contenu["personnages"]) if p["id"] == id_perso]
    if deja and not remplacer: raise ValueError(f"id {id_perso} déjà présent dans {chemin} (remplacer=True pour l'écraser)")
    for i in deja: contenu["personnages"].pop(i)
    contenu["personnages"].append(nouveau)
    contenu["nb_personnages"] = len(contenu["personnages"])
    os.makedirs(os.path.dirname(os.path.abspath(chemin)), exist_ok=True)
    with open(chemin, "w", encoding="utf-8") as f: json.dump(contenu, f, ensure_ascii=False, indent=2)
    print(f"Perso {nouveau['nom_complet']} (id {id_perso}) {'remplacé' if deja else 'ajouté'} dans {chemin} "
          f"({contenu['nb_personnages']} perso(s) dans le fichier)")
    return nouveau

In [18]:
CONTRAINTES_STATS = {"famille": "physique", "ATK": {"egal": 2800}, "SPD": {"pct_min": 80}}
IMPOSES_KIT = ["Dégâts continus"]

persos = creer_persos(moteur, moteur_kit, CONTRAINTES_STATS, IMPOSES_KIT, n=5, graine=0)
afficher_persos(persos)
print()
CHOIX = 0                                              # le perso à examiner
alertes = valider_perso(moteur, moteur_kit, persos[CHOIX])

# Pour valider un perso imaginé par toi (décommente et adapte) :
#MON_PERSO = dict(stats={"HP": 7800, "MP": 270, "ATK": 2800, "MATK": 2300, "DEF": 1500, "MDEF": 1450, "CRIT": 720, "SPD": 335},
#                  kit=["Rôle Break", "Dégâts continus"])
#valider_perso(moteur, moteur_kit, MON_PERSO)

--- stats ---
Tirages acceptés : 2272 / 20000 (11.4%) | vrais persos non free qui respectent déjà les contraintes : 0
  ⚠ moins de 10 vrais persos : la proposition s'appuie surtout sur le modèle, pas sur des exemples.
--- kit ---
Vrais persos non free qui ont déjà tous les éléments imposés : 47
Perso 0 (physique) : HP 8250 | MP 249 | ATK 2800 | MATK 2425 | DEF 1590 | MDEF 1515 | CRIT 784 | SPD 339
   kit : Attaque supplémentaire ; Bonus ATK (alliés) ; Dégâts + : ennemi en dégâts continus ; Dégâts continus ; Rôle Charge
Perso 1 (physique) : HP 7800 | MP 253 | ATK 2800 | MATK 2425 | DEF 1635 | MDEF 1635 | CRIT 741 | SPD 342
   kit : Attaque avec coût en HP ; Attaque supplémentaire ; Bonus ATK (alliés) ; Bonus résistance physique (alliés) ; Dégâts continus ; Recollection Field +1 ; Rôle Break ; Rôle Charge
Perso 2 (physique) : HP 7350 | MP 271 | ATK 2800 | MATK 2325 | DEF 1500 | MDEF 1275 | CRIT 806 | SPD 342
   kit : Dégâts + : ennemi en dégâts continus ; Dégâts continus ; Recollection F

### Export

In [19]:
nouveau = exporter_perso(
    persos[CHOIX], moteur, df_perso_complet,
    id_perso=999000002,                    # un id qui n'existe pas dans le jeu
    nom="Eugeo", titre="[pas bo]",
    element="feu", arme="Épée",           # les valeurs valides s'affichent dans le message d'erreur si tu te trompes
    chemin="../data/personage_new.json",
    parametres=dict(graine=0, contraintes_stats=CONTRAINTES_STATS, imposes_kit=IMPOSES_KIT))

Perso [pas bo] Eugeo (id 999000002) ajouté dans ../data/personage_new.json (2 perso(s) dans le fichier)
